<a href="https://colab.research.google.com/github/pop123-ux/HuggingFace-Project-Learning/blob/main/course/en/chapter5/Handling_big_datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Big data? 🤗 Datasets to the rescue!

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [1]:
!pip install datasets evaluate transformers[sentencepiece]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00


In [2]:
!pip install zstandard

In [7]:
from datasets import load_dataset

pubmed_dataset = load_dataset("ccdv/pubmed-summarization", split="train")
print(pubmed_dataset)

README.md:   0%|          | 0.00/3.80k [00:00<?, ?B/s]

section/train-00000-of-00005.parquet:   0%|          | 0.00/210M [00:00<?, ?B/s]

section/train-00001-of-00005.parquet:   0%|          | 0.00/208M [00:00<?, ?B/s]

section/train-00002-of-00005.parquet:   0%|          | 0.00/207M [00:00<?, ?B/s]

section/train-00003-of-00005.parquet:   0%|          | 0.00/211M [00:00<?, ?B/s]

...

section/test-00000-of-00001.parquet:   0%|          | 0.00/58.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/119924 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6633 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6658 [00:00<?, ? examples/s]

Dataset({
    features: ['article', 'abstract'],
    num_rows: 119924
})


In [8]:
pubmed_dataset

Dataset({
    features: ['article', 'abstract'],
    num_rows: 119924
})

In [9]:
pubmed_dataset[0]

{'article': "a recent systematic analysis showed that in 2011 , 314 ( 296 - 331 ) million children younger than 5 years were mildly , moderately or severely stunted and 258 ( 240 - 274 ) million were mildly , moderately or severely underweight in the developing countries . \n in iran a study among 752 high school girls in sistan and baluchestan showed prevalence of 16.2% , 8.6% and 1.5% , for underweight , overweight and obesity , respectively . \n the prevalence of malnutrition among elementary school aged children in tehran varied from 6% to 16% . \n anthropometric study of elementary school students in shiraz revealed that 16% of them suffer from malnutrition and low body weight . \n snack should have 300 - 400 kcal energy and could provide 5 - 10 g of protein / day . nowadays , school nutrition programs are running as the national programs , world - wide . national school lunch program in the united states \n there are also some reports regarding school feeding programs in developi

In [10]:
!pip install psutil

In [11]:
import psutil

print(f"RAM used: {psutil.Process().memory_info().rss / (1024 * 1024):2f} MB")

RAM used: 970.457031 MB


In [12]:
print(f"Dataset size in bytes: {pubmed_dataset.dataset_size}")
size_gb = pubmed_dataset.dataset_size / (1024**3)
print(f"Dataset size (cache file) : {size_gb:.2f} GB")

Dataset size in bytes: 2512943451
Dataset size (cache file) : 2.34 GB


In [13]:
import timeit

code_snippet = """batch_size = 1000

for idx in range(0, len(pubmed_dataset), batch_size):
    _ = pubmed_dataset[idx:idx + batch_size]
"""

time = timeit.timeit(stmt=code_snippet, number=1, globals=globals())
print(
    f"Iterated over {len(pubmed_dataset)} examples (about {size_gb:.1f} GB) in "
    f"{time:.1f}s, i.e. {size_gb/time:.3f} GB/s"
)

Iterated over 119924 examples (about 2.3 GB) in 4.7s, i.e. 0.495 GB/s


# Streaming datasets #

To enable dataset streaming we have to pass the streaming=True argument to the load_dataset() function

In [15]:
pubmed_dataset_streamed = load_dataset("ccdv/pubmed-summarization", split="train", streaming=True)

In [16]:
next(iter(pubmed_dataset_streamed))

{'article': "a recent systematic analysis showed that in 2011 , 314 ( 296 - 331 ) million children younger than 5 years were mildly , moderately or severely stunted and 258 ( 240 - 274 ) million were mildly , moderately or severely underweight in the developing countries . \n in iran a study among 752 high school girls in sistan and baluchestan showed prevalence of 16.2% , 8.6% and 1.5% , for underweight , overweight and obesity , respectively . \n the prevalence of malnutrition among elementary school aged children in tehran varied from 6% to 16% . \n anthropometric study of elementary school students in shiraz revealed that 16% of them suffer from malnutrition and low body weight . \n snack should have 300 - 400 kcal energy and could provide 5 - 10 g of protein / day . nowadays , school nutrition programs are running as the national programs , world - wide . national school lunch program in the united states \n there are also some reports regarding school feeding programs in developi

The elements from a streamed dataset can be processed on the fly using IterableDataset.map(), which is useful during training if you need to tokenize the inputs

In [22]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
tokenized_dataset = pubmed_dataset_streamed.map(lambda x: tokenizer(x['article']))
next(iter(tokenized_dataset))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (5243 > 512). Running this sequence through the model will result in indexing errors


{'article': "a recent systematic analysis showed that in 2011 , 314 ( 296 - 331 ) million children younger than 5 years were mildly , moderately or severely stunted and 258 ( 240 - 274 ) million were mildly , moderately or severely underweight in the developing countries . \n in iran a study among 752 high school girls in sistan and baluchestan showed prevalence of 16.2% , 8.6% and 1.5% , for underweight , overweight and obesity , respectively . \n the prevalence of malnutrition among elementary school aged children in tehran varied from 6% to 16% . \n anthropometric study of elementary school students in shiraz revealed that 16% of them suffer from malnutrition and low body weight . \n snack should have 300 - 400 kcal energy and could provide 5 - 10 g of protein / day . nowadays , school nutrition programs are running as the national programs , world - wide . national school lunch program in the united states \n there are also some reports regarding school feeding programs in developi

We can also shuffle a streamed dataset using IterableDataset.shuffle(), but unlike Dataset.shuffle(), this only shuffles the elements in a predefined buffer_size

In [24]:
shuffled_dataset = pubmed_dataset_streamed.shuffle(buffer_size=10_000, seed=42)
next(iter(shuffled_dataset))

{'article': 'there are two histological types of thyroid carcinoma arising from follicular cells , which are papillary carcinoma ( ptc ) and follicular carcinoma ( ftc ) . \n these two histological types are also called differentiated carcinoma ( dtc ) and analyzed as a single group for clinical studies investigating prognostic factors and prognosis of patients . \n ptc frequently metastasizes to the regional lymph nodes and can show a high incidence of significant extrathyroid extension to adjacent organs . \n in contrast , it is comparably rare for ftc to show these events , but ftc more frequently metastasizes to distant organs such as the lung , bone , and brain than ptc . \n therefore , analyses of prognostic factors and prognosis of patients should be performed separately for ptc and ftc . \n generally , ptc and ftc are indolent diseases and show good prognoses of patients , but when the lesion dedifferentiates , becoming undifferentiated carcinoma ( anaplastic carcinoma ) , the 

In this example, we selected a random example from the first 10,000 examples in the buffer. Once an example is accessed, its spot in the buffer is filled with the next example in the corpus (i.e., the 10,001st example in the case above). We can also select elements from a streamed dataset using the IterableDataset.take() and IterableDataset.skip() functions, which act in a similar way to Dataset.select()

In [25]:
# To select the first 5 examples in the PubMed Abstracts dataset:
dataset_head = pubmed_dataset_streamed.take(5)
list(dataset_head)

[{'article': "a recent systematic analysis showed that in 2011 , 314 ( 296 - 331 ) million children younger than 5 years were mildly , moderately or severely stunted and 258 ( 240 - 274 ) million were mildly , moderately or severely underweight in the developing countries . \n in iran a study among 752 high school girls in sistan and baluchestan showed prevalence of 16.2% , 8.6% and 1.5% , for underweight , overweight and obesity , respectively . \n the prevalence of malnutrition among elementary school aged children in tehran varied from 6% to 16% . \n anthropometric study of elementary school students in shiraz revealed that 16% of them suffer from malnutrition and low body weight . \n snack should have 300 - 400 kcal energy and could provide 5 - 10 g of protein / day . nowadays , school nutrition programs are running as the national programs , world - wide . national school lunch program in the united states \n there are also some reports regarding school feeding programs in develop

We can use the IterableDataset.skip() function to create training and validation splits from a shuffled dataset

In [26]:
train_dataset = shuffled_dataset.skip(1000)
validation_dataset = shuffled_dataset.take(1000)

## To do: ##
Use one of the large Common Crawl corpora like mc4 or oscar to create a streaming multilingual dataset that represents the spoken proportions of languages in a country of your choice.

For example, the four national languages in Switzerland are German, French, Italian, and Romansh, so you could try creating a Swiss corpus by sampling the Oscar subsets according to their spoken proportion.